# EvolveAI Brain v3.1 — Corrected Semantic Memory Build

This notebook is deliberately split into preflight, tests, source discovery, ingestion and retrieval. It does not train the SNN or modify Brain v2 checkpoints. It indexes only the examples actually saved in the episodic JSONL log.

In [ ]:
from pathlib import Path
import subprocess, sys, os, json
REPO = Path('/content/EvolveAI')
BRANCH = 'evolveai-maxgpu-one-shot'
if not (REPO / '.git').exists():
    if REPO.exists(): raise RuntimeError(f'{REPO} exists but is not a Git checkout. Rename or remove it, then rerun.')
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/swasthik-s/EvolveAI.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin',BRANCH],check=True)
sys.path.insert(0,str(REPO/'snn'))
os.chdir(REPO/'snn')
print('Repository:',REPO)
print('SNN package directory:',Path.cwd())
assert (Path.cwd()/'brain'/'semantic_memory.py').is_file(), 'semantic_memory.py missing from checkout'


In [ ]:
from brain.semantic_memory import SemanticMemory
import inspect
print('SemanticMemory import: OK')
print('Constructor:',inspect.signature(SemanticMemory))
print('Ingestion:',inspect.signature(SemanticMemory.ingest_episodic_jsonl))


In [ ]:
result = subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-p','test_semantic_memory.py','-v'],capture_output=True,text=True)
print(result.stdout)
print(result.stderr)
if result.returncode != 0: raise RuntimeError('Semantic memory unit tests failed; stop here and inspect the output above.')
print('Unit tests: PASS')


## Locate the Brain v2 episodic log

Mount Drive and discover the exact file instead of assuming one fixed run folder. If multiple logs exist, inspect their sizes and modification times before choosing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
SEARCH_ROOTS = [DRIVE_ROOT/'EvolveAI', DRIVE_ROOT]
found = {}
for root in SEARCH_ROOTS:
    if root.exists():
        for p in root.rglob('episodic_memory.jsonl'):
            if p.is_file(): found[str(p)] = p
    if found: break
candidates = sorted(found.values(),key=lambda p:p.stat().st_mtime,reverse=True)
print('Candidate episodic logs:',len(candidates))
for i,p in enumerate(candidates): print(i, '|',p,'|',p.stat().st_size,'bytes')
if not candidates: raise FileNotFoundError('No episodic_memory.jsonl found under Google Drive MyDrive. Check that the Brain v2 run saved its episodic log to Drive.')
EPISODIC = candidates[0]
if EPISODIC.stat().st_size == 0: raise ValueError(f'Episodic log is empty: {EPISODIC}')
MEMORY_DIR = DRIVE_ROOT/'EvolveAI'/'brain_v3_memory'
MEMORY_DIR.mkdir(parents=True,exist_ok=True)
print('\nSelected source:',EPISODIC)
print('Persistent memory destination:',MEMORY_DIR)


In [ ]:
with EPISODIC.open(encoding='utf-8') as f:
    first_line = next((line for line in f if line.strip()),None)
if first_line is None: raise ValueError('Episodic JSONL contains no non-empty records')
sample_event = json.loads(first_line)
required = {'domain','examples'}
missing = required - set(sample_event)
if missing: raise ValueError(f'Unexpected episodic schema; missing keys: {sorted(missing)}. Found: {sorted(sample_event)}')
if not isinstance(sample_event['examples'],list): raise TypeError('Event examples field must be a list')
print('Schema check: PASS')
print('Sample event keys:',sorted(sample_event))
print('Sample domain:',sample_event.get('domain'))
print('Sample examples:',len(sample_event['examples']))


## Build or resume semantic memory

The ingestion is idempotent for identical domain/source/text keys. Re-running this cell does not create duplicate documents. Existing memory files are loaded and updated.

In [ ]:
memory = SemanticMemory(MEMORY_DIR)
before = memory.stats()
print('Before:',before)
ingestion = memory.ingest_episodic_jsonl(EPISODIC)
print('\nINGESTION REPORT')
for key,value in ingestion.items(): print(f'{key}: {value}')
after = memory.stats()
print('\nAfter:',after)
if after['documents'] == 0: raise RuntimeError('No documents were indexed. Inspect episodic examples and the ingestion report.')
assert (MEMORY_DIR/'semantic_documents.jsonl').is_file()
assert (MEMORY_DIR/'knowledge_graph.json').is_file()
assert (MEMORY_DIR/'semantic_memory_manifest.json').is_file()
print('Persistence checks: PASS')


In [ ]:
queries = ['mathematics problem solution','language stories','news report','encyclopedic knowledge']
for query in queries:
    hits = memory.search(query,top_k=3)
    print('\nQUERY:',query,'| hits:',len(hits))
    for hit in hits:
        print('score:',hit['score'],'domain:',hit['domain'],'sources:',hit['source_ids'])
        print(hit['text'][:300])
if not any(memory.search(q,top_k=1) for q in queries):
    raise RuntimeError('Retrieval smoke test returned no hits. Check tokenization and indexed document content.')
print('\nRetrieval smoke test: PASS')


In [ ]:
print('TOP CONCEPT CANDIDATES')
ranked = sorted(memory.nodes.items(),key=lambda item:item[1]['document_count'],reverse=True)[:30]
for term,node in ranked: print(term,node['document_count'],node['domains'])
if ranked:
    term = ranked[0][0]
    print('\nCO-OCCURRENCE LINKS FOR:',term)
    for item in memory.related_concepts(term,15): print(item)
print('\nMANIFEST')
print((MEMORY_DIR/'semantic_memory_manifest.json').read_text(encoding='utf-8'))


## What this build does and does not do

- It indexes only up to two example snippets per domain event stored in Brain v2 episodic memory; it does not reconstruct all 65.5M training tokens.
- Candidate concepts are heuristic frequency-ranked words and adjacent bigrams.
- Graph edges mean two candidate terms co-occurred in one saved source record. They are not verified facts, causal relations or entailments.
- Retrieval is lexical BM25-style search, not embedding-based semantic retrieval and not answer generation.
- This notebook does not train or alter SNN weights. It creates persistent memory infrastructure for subsequent controlled evaluation.
- If any preflight, unit test, schema check or retrieval smoke test fails, stop at that cell; do not continue to later cells.